# Root cause from logs alone: turn raw events into a causal graph

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/logs-causal-graph-root-cause-esql).
Replays the incident, then walks the four steps that turn 4,688 raw log records into a causal
graph naming the service that failed first.

Requires Elasticsearch 9.1+ (step 3 uses `MIN()` over a keyword field). Runs in under a minute.

The incident: `checkout-api` calls `payment-gateway`, which calls `ledger-service`, which holds
a pool of 8 database connections. The pool saturates. `catalog-api` sits outside that path and
logs an ERROR on every image cache miss while returning 200 to everything, which is what makes
error counts point at the wrong service.

## Setup

Copy `.env.example` to `.env` and fill it in. The API key needs `manage_index_templates` on the
cluster, and `create_doc`, `auto_configure`, `read` and `manage` on `logs-apm.app.*` and
`traces-apm*`.

In [1]:
%pip install -q elasticsearch==9.5.0 python-dotenv==1.2.3

In [2]:
import os
import random
import uuid
import warnings
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from elasticsearch import Elasticsearch, helpers

# ES|QL applies a default LIMIT 1000 when a query does not set one. Every query here returns a
# handful of rows, so the notice is noise.
warnings.filterwarnings("ignore", message=".*No limit defined.*")

load_dotenv()

ES_URL = os.getenv("ELASTICSEARCH_URL")
API_KEY = os.getenv("ELASTICSEARCH_API_KEY")

missing = [
    n
    for n, v in (("ELASTICSEARCH_URL", ES_URL), ("ELASTICSEARCH_API_KEY", API_KEY))
    if not v
]
if missing:
    raise SystemExit(
        f"Missing in .env: {', '.join(missing)}. Copy .env.example to .env and fill it in."
    )

es_client = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)

# The API key needs manage_index_templates on the cluster, and create_doc plus auto_configure
# on logs-apm.app.* and traces-apm*. A key scoped to something narrower fails here with a 403.
try:
    version = es_client.info()["version"]["number"]
except Exception as error:
    raise SystemExit(
        f"Could not reach Elasticsearch with this API key: {error}\n"
        "Check ELASTICSEARCH_URL, and that the key has at least 'monitor' on the cluster."
    )

if tuple(int(part) for part in version.split(".")[:2]) < (9, 1):
    raise SystemExit(f"Elasticsearch {version} is too old. Step 3 needs 9.1 or later.")
print("connected to", version)

connected to 9.5.2


## 1. Create the data streams

The records land in `logs-apm.app.<service>-default` and `traces-apm-default`, the same data
streams the APM OTLP intake writes to, using ECS field names.

`service.name` and `message` are mapped as `keyword` because the queries group by them. Keys
with dots in them, which the services pass in the standard library logger's `extra` dict, arrive
as `labels.*` and `numeric_labels.*`, so those get dynamic templates.

In [3]:
LOGS_MAPPING = {
    "properties": {
        "@timestamp": {"type": "date"},
        "message": {"type": "keyword", "ignore_above": 1024},
        "service": {
            "properties": {
                "name": {"type": "keyword"},
                "environment": {"type": "keyword"},
            }
        },
        "trace": {"properties": {"id": {"type": "keyword"}}},
        "span": {"properties": {"id": {"type": "keyword"}}},
        "log": {"properties": {"level": {"type": "keyword"}}},
        "labels": {"type": "object", "dynamic": True},
        "numeric_labels": {"type": "object", "dynamic": True},
    },
    "dynamic_templates": [
        {"labels": {"path_match": "labels.*", "mapping": {"type": "keyword"}}},
        {
            "numeric_labels": {
                "path_match": "numeric_labels.*",
                "mapping": {"type": "double"},
            }
        },
    ],
}

TRACES_MAPPING = {
    "properties": {
        "@timestamp": {"type": "date"},
        "service": {
            "properties": {
                "name": {"type": "keyword"},
                "environment": {"type": "keyword"},
            }
        },
        "trace": {"properties": {"id": {"type": "keyword"}}},
        "transaction": {
            "properties": {
                "id": {"type": "keyword"},
                "name": {"type": "keyword"},
                "type": {"type": "keyword"},
                "duration": {"properties": {"us": {"type": "long"}}},
            }
        },
        "event": {"properties": {"outcome": {"type": "keyword"}}},
        "processor": {"properties": {"event": {"type": "keyword"}}},
    },
}

for name, pattern, mapping in [
    ("causal-graph-logs-apm", "logs-apm.app.*", LOGS_MAPPING),
    ("causal-graph-traces-apm", "traces-apm*", TRACES_MAPPING),
]:
    es_client.indices.put_index_template(
        name=name,
        index_patterns=[pattern],
        data_stream={},
        priority=500,
        template={
            "mappings": mapping,
            "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        },
    )
print("index templates created")

index templates created


## 2. Replay the incident

The article shows the four Flask services that produced this data, instrumented with EDOT
Python and logging through the standard library logger. Here we write the same records
directly, so the lab reproduces in seconds and the numbers come out the same for everyone.

Every failed checkout produces three ERROR records sharing one `trace.id`: `ledger-service`
first, then `payment-gateway`, then `checkout-api`. That shared `trace.id` is the whole
investigation. Two errors with the same one belong to the same request, so one may have caused
the other. Two errors with different ones are unrelated no matter how close their timestamps.

The millisecond spreads below are deliberate. ECS log timestamps have millisecond resolution
and these hops are local HTTP calls, so ties happen, and the article measures how often.

In [4]:
ENVIRONMENT = "obs-labs-causal-graph"
random.seed(20260726)

# The saturation window: 956 failed checkouts over four minutes.
FIRST = datetime(2026, 7, 26, 9, 1, 11, 665000, tzinfo=timezone.utc)
LAST = datetime(2026, 7, 26, 9, 5, 11, 464000, tzinfo=timezone.utc)
RUN_START = datetime(2026, 7, 26, 9, 0, 0, tzinfo=timezone.utc)
RUN_END = datetime(2026, 7, 26, 9, 6, 59, tzinfo=timezone.utc)

N_CASCADES = 956  # failed checkouts, three ERROR records each
N_CHECKOUT_OK = 4053  # successful checkouts
N_CATALOG_TX = 3345  # catalog requests, all returning 200
N_CATALOG_ERRORS = 1820  # of those, the ones that logged a cache miss

# One trace where all three errors share a millisecond, and 258 where two of them do.
# Alphabetically checkout-api < ledger-service < payment-gateway, so a payment/checkout tie
# still leaves ledger-service earliest. Only the three-way tie flips the answer.
spreads = [0] + [1] * 258 + [2] * 400
spreads += [3 + (i % 14) for i in range(N_CASCADES - len(spreads))]
COLLISION_INDEX = 477
spreads[0], spreads[COLLISION_INDEX] = spreads[COLLISION_INDEX], spreads[0]

COLLISION_TRACE = "f9b32dca62c1e142537413a047fb9b69"
COLLISION_AT = datetime(2026, 7, 26, 9, 3, 45, 139000, tzinfo=timezone.utc)

span = (LAST - FIRST).total_seconds()
times = [
    FIRST + timedelta(seconds=span * i / (N_CASCADES - 1)) for i in range(N_CASCADES)
]
times[COLLISION_INDEX] = COLLISION_AT


def ts(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.") + f"{dt.microsecond // 1000:03d}Z"


def hex_id(n=16):
    return uuid.uuid4().hex[:n]


def log_doc(service, at, message, labels, trace_id, numeric=None):
    source = {
        "@timestamp": ts(at),
        "service": {"name": service, "environment": ENVIRONMENT},
        "trace": {"id": trace_id},
        "span": {"id": hex_id()},
        "log": {"level": "ERROR"},
        "message": message,
        "labels": labels,
    }
    if numeric:
        source["numeric_labels"] = numeric
    return {
        "_index": f"logs-apm.app.{service}-default",
        "_op_type": "create",
        "_source": source,
    }


def tx_doc(service, at, name, outcome, trace_id, duration_us):
    return {
        "_index": "traces-apm-default",
        "_op_type": "create",
        "_source": {
            "@timestamp": ts(at),
            "service": {"name": service, "environment": ENVIRONMENT},
            "trace": {"id": trace_id},
            "processor": {"event": "transaction"},
            "event": {"outcome": outcome},
            "transaction": {
                "id": hex_id(),
                "name": name,
                "type": "request",
                "duration": {"us": duration_us},
            },
        },
    }


logs, transactions = [], []

for i in range(N_CASCADES):
    trace_id = COLLISION_TRACE if i == COLLISION_INDEX else hex_id(32)
    order_id = hex_id(12)
    t_ledger, spread = times[i], spreads[i]
    t_payment = t_ledger + timedelta(milliseconds=0 if spread == 0 else 1)
    t_checkout = t_ledger + timedelta(
        milliseconds=spread if spread > 1 else (0 if spread == 0 else 1)
    )

    # ledger-service names no upstream: nothing it depends on failed.
    logs.append(
        log_doc(
            "ledger-service",
            t_ledger,
            "connection pool exhausted, no connection available after 250ms",
            {"error_kind": "pool_timeout", "order_id": order_id},
            trace_id,
            {"db_connection_pool_size": 8, "db_connection_pool_available": 0},
        )
    )
    # The two callers each record which dependency they were waiting on.
    logs.append(
        log_doc(
            "payment-gateway",
            t_payment,
            "ledger rejected reservation with status 503, cannot authorize payment",
            {
                "error_kind": "authorization_failed",
                "upstream_service": "ledger-service",
                "upstream_status_code": "503",
                "order_id": order_id,
            },
            trace_id,
        )
    )
    logs.append(
        log_doc(
            "checkout-api",
            t_checkout,
            f"checkout failed for order {order_id}, payment authorization returned 502",
            {
                "error_kind": "checkout_failed",
                "upstream_service": "payment-gateway",
                "upstream_status_code": "502",
                "order_id": order_id,
            },
            trace_id,
        )
    )

    transactions += [
        tx_doc(
            "ledger-service", t_ledger, "POST /reserve", "failure", trace_id, 250000
        ),
        tx_doc(
            "payment-gateway", t_payment, "POST /authorize", "failure", trace_id, 252000
        ),
        tx_doc(
            "checkout-api", t_checkout, "POST /checkout", "failure", trace_id, 254000
        ),
    ]

print(f"{N_CASCADES} cascades, {len(logs)} error records")

956 cascades, 2868 error records


Now the traffic that succeeded, plus `catalog-api`. The decoy is the point: it returns 200 to
all 3,345 of its requests and still logs 1,820 errors, more than any service in the cascade.

In [5]:
run_span = (RUN_END - RUN_START).total_seconds()

for i in range(N_CHECKOUT_OK):
    at = RUN_START + timedelta(seconds=run_span * i / N_CHECKOUT_OK)
    trace_id = hex_id(32)
    transactions += [
        tx_doc("ledger-service", at, "POST /reserve", "success", trace_id, 15000),
        tx_doc("payment-gateway", at, "POST /authorize", "success", trace_id, 17000),
        tx_doc("checkout-api", at, "POST /checkout", "success", trace_id, 19000),
    ]

# Two retries against ledger, which is why its total runs slightly ahead of checkout's.
for i in range(2):
    transactions.append(
        tx_doc(
            "ledger-service",
            RUN_START + timedelta(seconds=i),
            "POST /reserve",
            "success",
            hex_id(32),
            15000,
        )
    )

for i in range(N_CATALOG_TX):
    at = RUN_START + timedelta(seconds=run_span * i / N_CATALOG_TX)
    trace_id = hex_id(32)
    transactions.append(
        tx_doc("catalog-api", at, "GET /product/{id}", "success", trace_id, 4000)
    )
    if i < N_CATALOG_ERRORS:
        logs.append(
            log_doc(
                "catalog-api",
                at,
                f"image cache miss for product sku-{i % 200:04d}, falling back to origin",
                {"error_kind": "image_cache_miss", "product_id": f"sku-{i % 200:04d}"},
                trace_id,
            )
        )

print(f"{len(logs)} error records, {len(transactions)} transactions")

4688 error records, 18374 transactions


Index everything. Re-running the notebook replaces the data rather than doubling it.

In [6]:
DATA_STREAMS = [
    "logs-apm.app.checkout-api-default",
    "logs-apm.app.payment-gateway-default",
    "logs-apm.app.ledger-service-default",
    "logs-apm.app.catalog-api-default",
    "traces-apm-default",
]

for stream in DATA_STREAMS:
    es_client.options(ignore_status=404).indices.delete_data_stream(name=stream)

indexed, errors = helpers.bulk(
    es_client, logs + transactions, chunk_size=2000, raise_on_error=False
)
es_client.indices.refresh(index="logs-apm.app.*,traces-apm*")
print(f"indexed {indexed} documents, {len(errors)} errors")

indexed 23062 documents, 0 errors


## 3. Step 1: rank services by failure rate, then by error volume

Every query is scoped to the run window and to this lab's environment.

Start with the metric an alert would fire on: the failed transaction rate per service.

In [7]:
WINDOW = (
    '@timestamp >= "2026-07-26T08:59:00.000Z" '
    'AND @timestamp < "2026-07-26T09:07:00.000Z" '
    f'AND service.environment == "{ENVIRONMENT}"'
)


def show(query, title):
    """Run an ES|QL query and print the result as a table."""
    response = es_client.esql.query(query=query, format="json")
    columns = [c["name"] for c in response["columns"]]
    rows = response["values"]
    print(f"\n{title}")
    print("-" * len(title))
    widths = [
        max([len(c)] + [len(str(r[i])) for r in rows]) for i, c in enumerate(columns)
    ]
    print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
    for row in rows:
        print(
            "  ".join(
                ("" if v is None else str(v)).ljust(w) for v, w in zip(row, widths)
            )
        )
    return rows


show(
    f"""
FROM traces-apm*
| WHERE {WINDOW} AND transaction.type IS NOT NULL
| STATS failed = COUNT(*) WHERE event.outcome == "failure", total = COUNT(*) BY service.name
| EVAL failed_pct = ROUND(100.0 * failed / total, 1)
| SORT failed_pct DESC
""",
    "Failure rate per service",
)


Failure rate per service
------------------------
failed  total  service.name     failed_pct
956     5009   checkout-api     19.1      
956     5011   ledger-service   19.1      
956     5009   payment-gateway  19.1      
0       3345   catalog-api      0.0       


[[956, 5009, 'checkout-api', 19.1],
 [956, 5011, 'ledger-service', 19.1],
 [956, 5009, 'payment-gateway', 19.1],
 [0, 3345, 'catalog-api', 0.0]]

Three services at an identical **19.1%**, 956 failures each. The rates match because in a
synchronous call chain every hop fails when the deepest hop fails. This names the affected
services and gives you no way to order them.

So try the other obvious ranking: who logged the most errors?

In [8]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW} AND log.level == "ERROR"
| STATS errors = COUNT(*), traces = COUNT_DISTINCT(trace.id) BY service.name
| SORT errors DESC
""",
    "Error log volume per service",
)


Error log volume per service
----------------------------
errors  traces  service.name   
1820    1820    catalog-api    
956     956     payment-gateway
956     956     ledger-service 
956     956     checkout-api   


[[1820, 1820, 'catalog-api'],
 [956, 956, 'payment-gateway'],
 [956, 956, 'ledger-service'],
 [956, 956, 'checkout-api']]

`catalog-api` leads with **1,820**, almost double anything else, and it returned 200 to every
request it served. Sorting by error count ranks services by how talkative their logging is,
which has nothing to do with where the failure started.

Both rankings are dead ends. Neither one produces an edge.

## 4. Step 2: group errors by trace

Instead of counting errors per service, count *services per trace*. How many distinct services
logged an error inside the same request?

In [9]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW} AND log.level == "ERROR"
| STATS services = COUNT_DISTINCT(service.name) BY trace.id
| STATS traces = COUNT(*) BY services
| SORT services ASC
""",
    "How many services logged an error in the same trace",
)


How many services logged an error in the same trace
---------------------------------------------------
traces  services
1820    1       
956     3       


[[1820, 1], [956, 3]]

The population splits cleanly in two:

- **1,820 traces** with an error from exactly one service. Local problems that never
  propagated, all of them `catalog-api`.
- **956 traces** with errors from three services. Each one is a cascade.

`catalog-api` appears in no cascade, so this single query removes the highest-volume service
from the investigation without anyone reading a log message.

## 5. Step 3: find the first service to fail in each trace

Within a cascade, the service that logged the first error is the candidate cause. ES|QL has no
window functions, so build a sortable string of timestamp plus service name, take the minimum
per trace, and slice the service name back out.

The formatted timestamp is fixed at 23 characters, so the earliest marker is also the
lexicographically smallest. `SUBSTRING(origin_marker, 25)` skips the timestamp and the
separator.

In [10]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW} AND log.level == "ERROR"
| EVAL marker = CONCAT(DATE_FORMAT("yyyy-MM-dd HH:mm:ss.SSS", @timestamp), "|", service.name)
| STATS origin_marker = MIN(marker), services = COUNT_DISTINCT(service.name) BY trace.id
| WHERE services > 1
| EVAL origin_service = SUBSTRING(origin_marker, 25)
| STATS cascade_traces = COUNT(*) BY origin_service
| SORT cascade_traces DESC
""",
    "Which service failed first, by timestamp",
)


Which service failed first, by timestamp
----------------------------------------
cascade_traces  origin_service
955             ledger-service
1               checkout-api  


[[955, 'ledger-service'], [1, 'checkout-api']]

`ledger-service` is the origin in **955 of 956** cascades. One trace disagrees. Rather than
wave that away, measure how trustworthy timestamp ordering is here.

In [11]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW} AND log.level == "ERROR"
| EVAL ms = DATE_FORMAT("yyyy-MM-dd HH:mm:ss.SSS", @timestamp)
| STATS spread_ms = DATE_DIFF("milliseconds", MIN(@timestamp), MAX(@timestamp)),
        distinct_ms = COUNT_DISTINCT(ms),
        services = COUNT_DISTINCT(service.name) BY trace.id
| WHERE services > 1
| STATS cascade_traces = COUNT(*),
        median_spread_ms = MEDIAN(spread_ms),
        max_spread_ms = MAX(spread_ms),
        traces_with_a_collision = COUNT(*) WHERE distinct_ms < services
""",
    "How far apart are the errors in a cascade",
)


How far apart are the errors in a cascade
-----------------------------------------
cascade_traces  median_spread_ms  max_spread_ms  traces_with_a_collision
956             2.0               16             259                    


[[956, 2.0, 16, 259]]

Median spread **2ms**, maximum 16ms, and **259 traces** (27%) contain at least one millisecond
collision. The hops are local HTTP calls and ECS log timestamps only go to milliseconds.

A collision is not automatically a wrong answer. Alphabetically `checkout-api` sorts before
`ledger-service`, which sorts before `payment-gateway`. When `ledger-service` and
`payment-gateway` land in the same millisecond, `MIN()` still returns `ledger-service`, which is
correct. Only a tie between `checkout-api` and `ledger-service` flips the result. That is why
259 traces collide and just one reports the wrong origin.

Timestamp ordering degrades on fast hops and on hosts whose clocks disagree by more than the gap
being measured. Treat it as a strong hint, not proof.

## 6. Step 4: build the causal edge list

The `upstream.service` field records which dependency the caller was waiting on, so every error
carrying it is already a directed edge. No timestamps involved.

In [12]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW}
    AND log.level == "ERROR" AND labels.upstream_service IS NOT NULL
| EVAL edge = CONCAT(labels.upstream_service, " -> ", service.name)
| STATS traces = COUNT_DISTINCT(trace.id), first_seen = MIN(@timestamp), last_seen = MAX(@timestamp) BY edge
| SORT traces DESC
""",
    "The causal edge list",
)


The causal edge list
--------------------
traces  first_seen                last_seen                 edge                             
956     2026-07-26T09:01:11.666Z  2026-07-26T09:05:11.465Z  ledger-service -> payment-gateway
956     2026-07-26T09:01:11.667Z  2026-07-26T09:05:11.469Z  payment-gateway -> checkout-api  


[[956,
  '2026-07-26T09:01:11.666Z',
  '2026-07-26T09:05:11.465Z',
  'ledger-service -> payment-gateway'],
 [956,
  '2026-07-26T09:01:11.667Z',
  '2026-07-26T09:05:11.469Z',
  'payment-gateway -> checkout-api']]

Two edges, each in all 956 cascades. This needs no clock synchronization and no tie-breaking,
and it agrees with the timestamp method on every trace including the one it got wrong.

The root of the graph is the node that appears as a source but never as a target: the service
that fails without naming an upstream.

In [13]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW} AND log.level == "ERROR"
| STATS errors = COUNT(*), traces = COUNT_DISTINCT(trace.id),
        names_an_upstream = COUNT_DISTINCT(labels.upstream_service) BY service.name
| WHERE names_an_upstream == 0
| SORT errors DESC
""",
    "Services that fail without naming an upstream",
)


Services that fail without naming an upstream
---------------------------------------------
errors  traces  names_an_upstream  service.name  
1820    1820    0                  catalog-api   
956     956     0                  ledger-service


[[1820, 1820, 0, 'catalog-api'], [956, 956, 0, 'ledger-service']]

Two candidates, `catalog-api` and `ledger-service`. Intersect them with the cascade participants
from step 2 and one remains: **`ledger-service`**.

Now read what it actually said, keeping the pool numbers so the evidence travels with the
message.

In [14]:
show(
    f"""
FROM logs-apm.app.*
| WHERE {WINDOW}
    AND log.level == "ERROR" AND service.name == "ledger-service"
| STATS events = COUNT(*), first_seen = MIN(@timestamp), last_seen = MAX(@timestamp)
        BY message, labels.error_kind,
           numeric_labels.db_connection_pool_size,
           numeric_labels.db_connection_pool_available
| SORT events DESC
""",
    "What ledger-service reported",
)


What ledger-service reported
----------------------------
events  first_seen                last_seen                 message                                                         labels.error_kind  numeric_labels.db_connection_pool_size  numeric_labels.db_connection_pool_available
956     2026-07-26T09:01:11.665Z  2026-07-26T09:05:11.464Z  connection pool exhausted, no connection available after 250ms  pool_timeout       8.0                                     0.0                                        


[[956,
  '2026-07-26T09:01:11.665Z',
  '2026-07-26T09:05:11.464Z',
  'connection pool exhausted, no connection available after 250ms',
  'pool_timeout',
  8.0,
  0.0]]

One distinct message across all 956 events, with a pool size of 8 and 0 connections available.
That is the root cause, stated by the service itself.

Finally, expand the one trace where all three errors share a millisecond. The timestamps cannot
order this cascade, and `upstream.service` recovers the chain anyway.

In [15]:
show(
    f"""
FROM logs-apm.app.*
| WHERE trace.id == "{COLLISION_TRACE}"
| KEEP @timestamp, service.name, log.level, message, labels.upstream_service
| SORT @timestamp ASC
""",
    "One cascade, expanded",
)


One cascade, expanded
---------------------
@timestamp                service.name     log.level  message                                                                     labels.upstream_service
2026-07-26T09:03:45.139Z  payment-gateway  ERROR      ledger rejected reservation with status 503, cannot authorize payment       ledger-service         
2026-07-26T09:03:45.139Z  ledger-service   ERROR      connection pool exhausted, no connection available after 250ms                                     
2026-07-26T09:03:45.139Z  checkout-api     ERROR      checkout failed for order b9033b2fb2d0, payment authorization returned 502  payment-gateway        


[['2026-07-26T09:03:45.139Z',
  'payment-gateway',
  'ERROR',
  'ledger rejected reservation with status 503, cannot authorize payment',
  'ledger-service'],
 ['2026-07-26T09:03:45.139Z',
  'ledger-service',
  'ERROR',
  'connection pool exhausted, no connection available after 250ms',
  None],
 ['2026-07-26T09:03:45.139Z',
  'checkout-api',
  'ERROR',
  'checkout failed for order b9033b2fb2d0, payment authorization returned 502',
  'payment-gateway']]

`labels.upstream_service` reads `(null)` for `ledger-service`, `ledger-service` for
`payment-gateway`, and `payment-gateway` for `checkout-api`. Walk those edges backwards and the
node you cannot walk back from is the cause.

## What to take away

Three things make this work, and only one costs any effort:

- **`trace.id` on every record.** OpenTelemetry auto-instrumentation supplies it for logs
  emitted inside an active span. Access logs and background threads fall outside that scope.
- **Structured fields instead of formatted strings.** `error.kind` as a field lets you count
  failure modes. The same words inside a sentence do not.
- **The upstream dependency named in the error.** One key, `upstream.service`, on the errors
  your service logs when a dependency fails.

## Cleanup

In [16]:
for stream in DATA_STREAMS:
    es_client.options(ignore_status=404).indices.delete_data_stream(name=stream)

for template in ("causal-graph-logs-apm", "causal-graph-traces-apm"):
    es_client.options(ignore_status=404).indices.delete_index_template(name=template)

print("cleanup complete")

cleanup complete
